# Notebook-first application walkthrough

**Problem / objective:** Forecast energy demand against strong seasonal baselines using time-ordered validation.

**Decision / solution:** Use forecast intervals and seasonal error slices to support capacity planning while identifying periods requiring extra reserve.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'energy_demand_forecasting'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Use forecast intervals and seasonal error slices to support capacity planning while identifying periods requiring extra reserve.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# 05 — Energy Demand Forecasting with TensorFlow

**Goal:** forecast the next 14 days of German electricity consumption for capacity planning, compare a TensorFlow LSTM with honest seasonal baselines, quantify uncertainty, and package a tested model artifact.

[Open in Google Colab](https://colab.research.google.com/github/Jorgoluka100/uni_projects/blob/main/05_Energy_Demand_Forecasting_with_TensorFlow.ipynb)

This notebook uses real public data, preserves chronological order, fits preprocessing on training data only, evaluates once on an untouched test period, and generates every reported metric during execution.

## Business framing

Grid and operations teams need forward demand estimates to schedule supply, maintenance and reserve capacity. The prediction target is daily electricity consumption for the next 14 days. Success means beating both a last-value baseline and a 7-day seasonal baseline on test MAE. Forecast intervals are included for planning—not as guaranteed bounds.

In [1]:
import json, os, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

SEED=42
os.environ['PYTHONHASHSEED']=str(SEED); random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
ROOT=Path('/content/energy_forecast' if Path('/content').exists() else './energy_forecast'); ROOT.mkdir(parents=True,exist_ok=True)
print({'tensorflow':tf.__version__,'seed':SEED,'devices':[d.device_type for d in tf.config.list_physical_devices()]})

{'tensorflow': '2.20.0', 'seed': 42, 'devices': ['CPU']}


## 1. Real dataset and data contract

Source: [Open Power System Data time-series package](https://data.open-power-system-data.org/time_series/) mirrored in the project-maintainer repository. It contains German daily electricity consumption, wind and solar generation from 2006–2017. This project uses `Consumption` as the target and weather-linked generation columns only as historical context where available. Review the upstream data package and licence before commercial reuse.

In [2]:
DATA_URL='https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv'
df=pd.read_csv(DATA_URL,parse_dates=['Date']).sort_values('Date').set_index('Date')
df=df.rename(columns={'Consumption':'consumption','Wind':'wind','Solar':'solar','Wind+Solar':'wind_solar'})
required=['consumption']; assert set(required)<=set(df.columns)
assert df.index.is_monotonic_increasing and df.index.is_unique
assert (df['consumption']>0).all()
expected=pd.date_range(df.index.min(),df.index.max(),freq='D')
missing_dates=expected.difference(df.index)
print({'rows':len(df),'start':str(df.index.min().date()),'end':str(df.index.max().date()),'missing_dates':len(missing_dates),'missing_target':int(df.consumption.isna().sum())})
display(df.head()); display(df.describe().T)
assert len(df)==4383 and len(missing_dates)==0 and df.consumption.notna().all()

{'rows': 4383, 'start': '2006-01-01', 'end': '2017-12-31', 'missing_dates': 0, 'missing_target': 0}
            consumption  wind  solar  wind_solar
Date                                            
2006-01-01     1069.184   NaN    NaN         NaN
2006-01-02     1380.521   NaN    NaN         NaN
2006-01-03     1442.533   NaN    NaN         NaN
2006-01-04     1457.217   NaN    NaN         NaN
2006-01-05     1477.131   NaN    NaN         NaN
              count         mean         std  ...       50%         75%       max
consumption  4383.0  1338.675836  165.775710  ...  1367.123  1457.76100  1709.568
wind         2920.0   164.814173  143.692732  ...   119.098   217.90025   826.278
solar        2188.0    89.258695   58.550099  ...    86.407   135.07150   241.580
wind_solar   2187.0   272.663481  146.319884  ...   240.991   338.98800   851.556

[4 rows x 8 columns]


## 2. Exploration without future leakage

In [3]:
fig,axs=plt.subplots(2,1,figsize=(13,8))
df.consumption.plot(ax=axs[0],color='#2457C5',lw=.8,title='German daily electricity consumption')
df.consumption.loc['2016':].plot(ax=axs[1],color='#E36A2E',lw=1,title='Recent two-year view')
for ax in axs: ax.set_ylabel('GWh'); ax.set_xlabel('');
plt.tight_layout(); plt.show()
calendar=pd.DataFrame({'consumption':df.consumption,'weekday':df.index.day_name(),'month':df.index.month})
fig,axs=plt.subplots(1,2,figsize=(13,4)); sns.boxplot(data=calendar,x='weekday',y='consumption',order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],ax=axs[0]); axs[0].tick_params(axis='x',rotation=35); sns.boxplot(data=calendar,x='month',y='consumption',ax=axs[1]); plt.tight_layout(); plt.show()

## 3. Chronological split and leakage-safe windows

The first 70% trains the model, the next 15% selects training duration and the final 15% is untouched until evaluation. The scaler is fitted only on the training period. Each sample uses the previous 60 days to predict the next 14 days.

In [4]:
LOOKBACK,HORIZON=60,14
n=len(df); train_end=int(.70*n); val_end=int(.85*n)
values=df[['consumption']].to_numpy('float32')
scaler=StandardScaler().fit(values[:train_end]); scaled=scaler.transform(values).astype('float32')

def make_windows(first_target,last_target):
    X,y,dates=[],[],[]
    for i in range(max(first_target,LOOKBACK),last_target-HORIZON+1):
        X.append(scaled[i-LOOKBACK:i]); y.append(scaled[i:i+HORIZON,0]); dates.append(df.index[i])
    return np.asarray(X),np.asarray(y),np.asarray(dates)
X_train,y_train,d_train=make_windows(0,train_end)
X_val,y_val,d_val=make_windows(train_end,val_end)
X_test,y_test,d_test=make_windows(val_end,n)
assert d_train.max()<d_val.min()<d_test.min()
print({'train':X_train.shape,'validation':X_val.shape,'test':X_test.shape,'train_last_target_start':str(pd.Timestamp(d_train.max()).date()),'test_first_target_start':str(pd.Timestamp(d_test.min()).date())})

{'train': (2995, 60, 1), 'validation': (644, 60, 1), 'test': (645, 60, 1), 'train_last_target_start': '2014-05-13', 'test_first_target_start': '2016-03-14'}


## 4. Baselines that the neural network must beat

In [5]:
def invert(x): return scaler.inverse_transform(np.asarray(x).reshape(-1,1)).reshape(np.asarray(x).shape)
actual_test=invert(y_test)
last_value_scaled=np.repeat(X_test[:,-1,0,None],HORIZON,axis=1)
seasonal_scaled=np.stack([X_test[:,LOOKBACK-7+(h%7),0] for h in range(HORIZON)],axis=1)
last_value=invert(last_value_scaled); seasonal=invert(seasonal_scaled)
def metrics(y,p):
    return {'MAE':mean_absolute_error(y.ravel(),p.ravel()),'RMSE':mean_squared_error(y.ravel(),p.ravel())**.5,'MAPE':np.mean(np.abs((y-p)/y))*100}
baseline_results=pd.DataFrame({'Last value':metrics(actual_test,last_value),'7-day seasonal':metrics(actual_test,seasonal)}).T
display(baseline_results.round(2))

                   MAE    RMSE   MAPE
Last value      146.62  197.13  11.23
7-day seasonal   53.18   93.39   3.96


## 5. TensorFlow LSTM model

In [6]:
tf.keras.backend.clear_session()
model=tf.keras.Sequential([
    tf.keras.layers.Input((LOOKBACK,1)),
    tf.keras.layers.Conv1D(32,5,padding='causal',activation='relu'),
    tf.keras.layers.LSTM(48,dropout=.15),
    tf.keras.layers.Dense(48,activation='relu'),
    tf.keras.layers.Dropout(.15),
    tf.keras.layers.Dense(HORIZON)
],name='energy_forecaster')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mae',metrics=['mae'])
callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=6,min_delta=1e-4,restore_best_weights=True),tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=3,factor=.5,min_lr=1e-5)]
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=40,batch_size=64,shuffle=False,callbacks=callbacks,verbose=0)
print({'epochs_trained':len(history.history['loss']),'best_validation_mae_scaled':min(history.history['val_loss'])})
pd.DataFrame(history.history)[['loss','val_loss']].plot(figsize=(9,4),title='Training history'); plt.ylabel('Scaled MAE'); plt.xlabel('Epoch'); plt.show()

{'epochs_trained': 40, 'best_validation_mae_scaled': 0.2646414339542389}


## 6. Untouched test evaluation

In [7]:
pred_test=invert(model.predict(X_test,verbose=0))
results=pd.DataFrame({'Last value':metrics(actual_test,last_value),'7-day seasonal':metrics(actual_test,seasonal),'TensorFlow LSTM':metrics(actual_test,pred_test)}).T
best_baseline=baseline_results.MAE.min(); improvement=1-results.loc['TensorFlow LSTM','MAE']/best_baseline
display(results.round(2)); print({'mae_improvement_vs_best_baseline_pct':round(improvement*100,2)})

horizon_rows=[]
for h in range(HORIZON):
    row=metrics(actual_test[:,h],pred_test[:,h]); row['horizon_day']=h+1; row['seasonal_MAE']=mean_absolute_error(actual_test[:,h],seasonal[:,h]); horizon_rows.append(row)
horizon_metrics=pd.DataFrame(horizon_rows)
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(horizon_metrics.horizon_day,horizon_metrics.MAE,marker='o',label='LSTM'); ax.plot(horizon_metrics.horizon_day,horizon_metrics.seasonal_MAE,marker='o',label='Seasonal'); ax.set(xlabel='Forecast day',ylabel='MAE (GWh)',title='Error by forecast horizon'); ax.legend(); plt.show()
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(pd.date_range(d_test[0],periods=HORIZON),actual_test[0],marker='o',label='Actual'); ax.plot(pd.date_range(d_test[0],periods=HORIZON),pred_test[0],marker='o',label='Forecast'); ax.plot(pd.date_range(d_test[0],periods=HORIZON),seasonal[0],ls='--',label='Seasonal baseline'); ax.legend(); ax.set(title='First untouched test forecast',ylabel='GWh'); plt.show()

                    MAE    RMSE   MAPE
Last value       146.62  197.13  11.23
7-day seasonal    53.18   93.39   3.96
TensorFlow LSTM   43.51   71.31   3.26
{'mae_improvement_vs_best_baseline_pct': np.float64(18.17)}


## 7. Calibrated uncertainty intervals

Absolute validation residuals calibrate a distribution-free interval for each horizon. This does not assume Gaussian errors. The test set is used only to report coverage and width—not to choose the interval.

In [8]:
pred_val=invert(model.predict(X_val,verbose=0)); actual_val=invert(y_val)
alpha=.10
q=np.quantile(np.abs(actual_val-pred_val),1-alpha,axis=0,method='higher')
lower=pred_test-q; upper=pred_test+q
coverage=np.mean((actual_test>=lower)&(actual_test<=upper)); avg_width=np.mean(upper-lower)
interval_metrics={'nominal_coverage':float(1-alpha),'empirical_test_coverage':float(coverage),'average_interval_width_gwh':float(avg_width)}
print(json.dumps(interval_metrics,indent=2))
fig,ax=plt.subplots(figsize=(12,4)); dates=pd.date_range(d_test[0],periods=HORIZON); ax.plot(dates,actual_test[0],marker='o',label='Actual'); ax.plot(dates,pred_test[0],marker='o',label='Forecast'); ax.fill_between(dates,lower[0],upper[0],alpha=.25,label='90% calibrated interval'); ax.legend(); ax.set(title='Forecast with validation-calibrated interval',ylabel='GWh'); plt.show()

{
  "nominal_coverage": 0.9,
  "empirical_test_coverage": 0.8837209302325582,
  "average_interval_width_gwh": 169.7863006591797
}


## 8. Operational error analysis

In [9]:
test_rows=pd.DataFrame({'forecast_start':pd.to_datetime(d_test),'actual_mean':actual_test.mean(1),'predicted_mean':pred_test.mean(1),'MAE':np.mean(np.abs(actual_test-pred_test),axis=1),'actual_peak':actual_test.max(1),'predicted_peak':pred_test.max(1)})
test_rows['season']=test_rows.forecast_start.dt.month.map({12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',6:'Summer',7:'Summer',8:'Summer',9:'Autumn',10:'Autumn',11:'Autumn'})
display(test_rows.groupby('season').MAE.agg(['count','mean','median','max']).round(2))
display(test_rows.nlargest(10,'MAE'))
fig,ax=plt.subplots(figsize=(9,4)); sns.boxplot(data=test_rows,x='season',y='MAE',order=['Winter','Spring','Summer','Autumn'],ax=ax); ax.set_title('Forecast error by season'); plt.show()

        count       mean     median         max
season                                         
Autumn    182  48.299999  45.130001  170.080002
Spring    171  45.040001  44.419998   94.519997
Summer    184  24.440001  20.670000   71.769997
Winter    108  65.529999  53.279999  161.050003
    forecast_start  actual_mean  ...  predicted_peak  season
233     2016-11-02  1475.218018  ...     1409.120605  Autumn
644     2017-12-18  1321.803833  ...     1596.948364  Winter
292     2016-12-31  1482.141968  ...     1464.587280  Winter
643     2017-12-17  1335.382690  ...     1595.335693  Winter
598     2017-11-02  1457.651611  ...     1404.778198  Autumn
597     2017-11-01  1439.591064  ...     1410.823730  Autumn
642     2017-12-16  1347.341919  ...     1576.190918  Winter
294     2017-01-02  1506.403442  ...     1457.926147  Winter
291     2016-12-30  1459.438477  ...     1432.238159  Winter
281     2016-12-20  1318.408447  ...     1569.936768  Winter

[10 rows x 7 columns]


## 9. Save the model and verify inference

In [10]:
model_path=ROOT/'energy_demand_forecaster.keras'; model.save(model_path)
reloaded=tf.keras.models.load_model(model_path)
original=model.predict(X_test[:8],verbose=0); restored=reloaded.predict(X_test[:8],verbose=0)
max_delta=float(np.max(np.abs(original-restored))); assert max_delta<1e-6
print({'artifact':str(model_path),'size_mb':model_path.stat().st_size/1e6,'max_prediction_delta':max_delta})

{'artifact': 'energy_forecast/energy_demand_forecaster.keras', 'size_mb': 0.262159, 'max_prediction_delta': 0.0}


## 10. Acceptance tests, model card and CV evidence

In [11]:
checks={
 'real_dataset_rows':len(df)==4383,
 'continuous_daily_index':len(missing_dates)==0,
 'target_complete':df.consumption.notna().all(),
 'chronological_boundaries':d_train.max()<d_val.min()<d_test.min(),
 'train_only_scaler':np.isclose(scaler.mean_[0],values[:train_end,0].mean(),rtol=1e-5),
 'prediction_shape':pred_test.shape==actual_test.shape,
 'finite_predictions':np.isfinite(pred_test).all(),
 'probabilistic_order':np.all(lower<=pred_test) and np.all(pred_test<=upper),
 'model_beats_last_value':results.loc['TensorFlow LSTM','MAE']<results.loc['Last value','MAE'],
 'model_artifact_verified':model_path.exists() and max_delta<1e-6
}
display(pd.Series(checks,name='passed').to_frame()); assert all(checks.values())
run_summary={'test_start':str(pd.Timestamp(d_test.min()).date()),'test_end':str(df.index.max().date()),'test_windows':int(len(X_test)),'tensorflow':{k:float(v) for k,v in results.loc['TensorFlow LSTM'].items()},'seasonal_baseline':{k:float(v) for k,v in results.loc['7-day seasonal'].items()},'improvement_vs_best_baseline_pct':float(improvement*100),**interval_metrics}
print('RUN-DERIVED SUMMARY'); print(json.dumps(run_summary,indent=2))
print(f"CV bullet: Built a leakage-safe TensorFlow demand forecaster on {len(df):,} days of real German electricity data; achieved {results.loc['TensorFlow LSTM','MAE']:.1f} GWh test MAE ({improvement:.1%} improvement over the strongest baseline) with chronological validation, horizon diagnostics, calibrated intervals and verified model reload.")

model_card={'model':'Conv1D + LSTM multi-horizon forecaster','target':'next 14 days of German electricity consumption','intended_use':'portfolio demonstration and capacity-planning decision support','not_for':'automatic grid control or safety-critical dispatch','training_period':f"{df.index.min().date()} to {df.index[train_end-1].date()}",'metrics':run_summary,'limitations':['dataset ends in 2017 and may not represent current demand','univariate history omits weather, price, policy and structural changes','interval calibration can degrade under distribution shift','operational use requires rolling backtests, drift monitoring and retraining']}
print(json.dumps(model_card,indent=2))

                          passed
real_dataset_rows           True
continuous_daily_index      True
target_complete             True
chronological_boundaries    True
train_only_scaler           True
prediction_shape            True
finite_predictions          True
probabilistic_order         True
model_beats_last_value      True
model_artifact_verified     True
RUN-DERIVED SUMMARY
{
  "test_start": "2016-03-14",
  "test_end": "2017-12-31",
  "test_windows": 645,
  "tensorflow": {
    "MAE": 43.51486587524414,
    "RMSE": 71.30503288579109,
    "MAPE": 3.256746768951416
  },
  "seasonal_baseline": {
    "MAE": 53.17710494995117,
    "RMSE": 93.3877114423386,
    "MAPE": 3.9591517448425293
  },
  "improvement_vs_best_baseline_pct": 18.169923097169104,
  "nominal_coverage": 0.9,
  "empirical_test_coverage": 0.8837209302325582,
  "average_interval_width_gwh": 169.7863006591797
}
CV bullet: Built a leakage-safe TensorFlow demand forecaster on 4,383 days of real German electricity data; achie

## Conclusion

The neural forecast is accepted only if it beats simple operational baselines on the untouched future period. Horizon and seasonal diagnostics show where errors concentrate, while validation-calibrated intervals express planning uncertainty. Production work should add weather forecasts and calendar events, use rolling-origin backtesting, monitor residual drift and coverage, and retrain on current licensed data.

# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Robustness, slices and decision analysis

A model or pipeline is useful only when we know where it works, where it fails and what action follows. This section adds direct slice analysis, sensitivity checks and a compact decision memo from the evidence already produced by the project.


In [ ]:
# Quantile slices for important numeric variables
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    quantile_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce')
        valid = values.dropna()
        if len(valid) < 20 or valid.nunique() < 5:
            continue
        quantiles = valid.quantile([0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.99])
        for q, value in quantiles.items():
            quantile_rows.append({'feature':col, 'quantile':q, 'value':float(value)})
    quantile_table = pd.DataFrame(quantile_rows)
    if len(quantile_table):
        display(quantile_table.pivot(index='feature', columns='quantile', values='value').round(4))
        for col in quantile_table['feature'].unique()[:6]:
            view = quantile_table[quantile_table['feature']==col]
            plt.figure(figsize=(7,4))
            plt.plot(view['quantile'], view['value'], marker='o')
            plt.xlabel('Quantile')
            plt.ylabel(col)
            plt.title(f'Quantile profile: {col}')
            plt.tight_layout()
            plt.show()
else:
    print('Quantile slices become available after the project dataset is materialised.')


In [ ]:
# Missingness and duplication sensitivity
if df is not None and len(df):
    missing_by_row = df.isna().sum(axis=1)
    print('Rows with any missing value:', int((missing_by_row>0).sum()))
    print('Rows with 2+ missing values:', int((missing_by_row>=2).sum()))
    print('Exact duplicate rows:', int(df.duplicated().sum()))
    if missing_by_row.max() > 0:
        plt.figure(figsize=(7,4))
        missing_by_row.value_counts().sort_index().plot(kind='bar')
        plt.title('Missing cells per row')
        plt.xlabel('Missing cells')
        plt.ylabel('Rows')
        plt.tight_layout()
        plt.show()
    duplicated = df.duplicated(keep=False)
    if duplicated.any():
        display(df.loc[duplicated].head(20))
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:10]
    robust_rows = []
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 20:
            continue
        median = values.median()
        mad = np.median(np.abs(values-median))
        robust_z = 0.6745*(values-median)/(mad if mad else 1.0)
        robust_rows.append({'feature':col, 'median':median, 'mad':mad, 'robust_outliers_abs_z_gt_3_5':int((np.abs(robust_z)>3.5).sum())})
    robust_outliers = pd.DataFrame(robust_rows).sort_values('robust_outliers_abs_z_gt_3_5', ascending=False) if robust_rows else pd.DataFrame()
    if len(robust_outliers):
        display(robust_outliers.round(4))


In [ ]:
# Concentration / imbalance analysis for important categorical dimensions
if df is not None and len(df):
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 50][:10]
    concentration_rows = []
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts()
        shares = counts / counts.sum()
        hhi = float((shares**2).sum())
        concentration_rows.append({'feature':col, 'categories':len(counts), 'largest_share':float(shares.iloc[0]), 'top3_share':float(shares.head(3).sum()), 'hhi':hhi})
    concentration = pd.DataFrame(concentration_rows).sort_values('hhi', ascending=False) if concentration_rows else pd.DataFrame()
    if len(concentration):
        display(concentration.round(4))
        plt.figure(figsize=(9,4))
        plt.bar(concentration['feature'], concentration['largest_share'])
        plt.ylabel('Largest category share')
        plt.title('Category concentration / imbalance')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()


In [ ]:
# Rank all retained scalar metrics and highlight likely success/risk signals
metric_records = []
for path in json_files[:60]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int,float)) and not isinstance(value,bool) and np.isfinite(value):
            metric_records.append({'file':path.name, 'metric':prefix, 'value':float(value)})
all_metrics = pd.DataFrame(metric_records)
if len(all_metrics):
    signal_pattern = 'accuracy|f1|auc|precision|recall|r2|rmse|mae|loss|coverage|review|drift|psi|brier|calibration|revenue|cost|effect|lift|latency|row|reject|duplicate'
    decision_metrics = all_metrics[all_metrics['metric'].str.contains(signal_pattern, case=False, regex=True)].copy()
    if not len(decision_metrics):
        decision_metrics = all_metrics.copy()
    decision_metrics = decision_metrics.drop_duplicates(['file','metric']).reset_index(drop=True)
    display(decision_metrics.head(60).round(6))
    rate_like = decision_metrics[decision_metrics['metric'].str.contains('accuracy|f1|auc|precision|recall|coverage|rate|r2', case=False, regex=True)]
    if len(rate_like):
        bounded = rate_like[(rate_like['value']>=-1)&(rate_like['value']<=1)].head(30)
        if len(bounded):
            plt.figure(figsize=(10,max(5,0.3*len(bounded))))
            plt.barh(range(len(bounded)), bounded['value'])
            plt.yticks(range(len(bounded)), bounded['file']+' :: '+bounded['metric'])
            plt.xlim(min(-0.05,bounded['value'].min()-0.05),1.05)
            plt.title('Retained rate / quality metrics')
            plt.tight_layout()
            plt.show()
    error_like = decision_metrics[decision_metrics['metric'].str.contains('rmse|mae|loss|error|latency|drift|psi|brier', case=False, regex=True)]
    if len(error_like):
        display(error_like.sort_values('value', ascending=False).head(30).round(6))
else:
    print('No retained scalar JSON metrics are available yet.')


In [ ]:
# Inspect artifact sizes — a quick engineering sanity check
artifact_rows = []
for base in [PROJECT/'artifacts', PROJECT/'results', PROJECT/'outputs', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in base.rglob('*'):
        if path.is_file():
            artifact_rows.append({'file':str(path.relative_to(ROOT)) if ROOT in path.parents else str(path), 'suffix':path.suffix.lower(), 'size_kb':path.stat().st_size/1024})
artifacts_df = pd.DataFrame(artifact_rows).sort_values('size_kb', ascending=False) if artifact_rows else pd.DataFrame()
if len(artifacts_df):
    display(artifacts_df.head(40).round(2))
    by_type = artifacts_df.groupby('suffix', as_index=False).agg(files=('file','size'), total_kb=('size_kb','sum')).sort_values('total_kb', ascending=False)
    display(by_type.round(2))
    plt.figure(figsize=(8,4))
    plt.bar(by_type['suffix'].replace('', '<none>'), by_type['total_kb'])
    plt.ylabel('Total KB')
    plt.title('Retained evidence by file type')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No retained artifacts/results found.')


In [ ]:
# Threshold / coverage trade-off when a result table contains confidence or probability
for path, table in result_tables:
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in ('confidence','probability','proba','score','risk'))]
    correct_cols = [c for c in table.columns if 'correct' in str(c).lower()]
    if not conf_cols or not len(table):
        continue
    confidence = pd.to_numeric(table[conf_cols[0]], errors='coerce')
    valid_conf = confidence.notna()
    if valid_conf.sum() < 20:
        continue
    trade_rows = []
    for threshold in np.linspace(float(confidence[valid_conf].quantile(0.10)), float(confidence[valid_conf].quantile(0.90)), 9):
        accepted = valid_conf & (confidence >= threshold)
        row = {'threshold':float(threshold), 'coverage':float(accepted.mean()), 'review_rate':float((valid_conf & ~accepted).sum()/valid_conf.sum()), 'accepted_rows':int(accepted.sum())}
        if correct_cols:
            correctness = table[correct_cols[0]].astype(bool)
            row['accepted_accuracy'] = float(correctness[accepted].mean()) if accepted.any() else np.nan
        trade_rows.append(row)
    trade = pd.DataFrame(trade_rows)
    print('Trade-off table from', path.name, 'using', conf_cols[0])
    display(trade.round(4))
    plt.figure(figsize=(8,4))
    plt.plot(trade['threshold'], trade['coverage'], marker='o', label='coverage')
    if 'accepted_accuracy' in trade:
        plt.plot(trade['threshold'], trade['accepted_accuracy'], marker='o', label='accepted accuracy')
    plt.xlabel('Threshold')
    plt.ylabel('Rate')
    plt.title(f'Threshold trade-off — {path.name}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    break


In [ ]:
# Produce a concise evidence-backed decision memo inside the notebook
project_summary = {
    'project': PROJECT_SLUG,
    'local_data_or_evidence_files': int(len(candidate_files)),
    'result_tables': int(len(result_tables)),
    'json_evidence_files': int(len(json_files)),
    'visual_evidence_files': int(len(png_files)),
    'has_tests': bool((PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))),
    'has_readme': bool((PROJECT/'README.md').exists()),
}
if df is not None:
    project_summary.update({'inspected_rows':int(len(df)), 'inspected_columns':int(df.shape[1]), 'duplicate_rows':int(df.duplicated().sum()), 'missing_cells':int(df.isna().sum().sum())})
summary_table = pd.DataFrame({'item':list(project_summary.keys()), 'value':list(project_summary.values())})
display(summary_table)
print('DECISION PRINCIPLE')
print('1. Use the measured evidence above, not model complexity, to choose the final approach.')
print('2. Inspect the worst slices/failures before making a business or operational recommendation.')
print('3. Keep uncertain, novel or high-impact cases on a review/escalation path where appropriate.')
print('4. Treat the documented limitations as part of the solution, not as boilerplate.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

from src.evaluation import (
    calibrate_residual_intervals,
    forecast_metrics,
    interval_metrics,
    last_value_baseline,
    seasonal_weekly_baseline,
)

ROOT = Path(__file__).resolve().parent
EVIDENCE = ROOT / "results" / "verified_metrics.json"


def self_test() -> None:
    X = np.arange(2 * 60, dtype=float).reshape(2, 60, 1) + 100.0
    last = last_value_baseline(X)
    weekly = seasonal_weekly_baseline(X)
    assert last.shape == (2, 14)
    assert weekly.shape == (2, 14)
    assert np.all(last[:, 0] == X[:, -1, 0])
    assert np.all(weekly[:, 0] == X[:, 53, 0])

    actual = np.array([[10.0, 20.0], [12.0, 18.0]])
    predicted = np.array([[11.0, 18.0], [11.0, 20.0]])
    metrics = forecast_metrics(actual, predicted)
    assert metrics["mae"] == 1.5
    radius = calibrate_residual_intervals(actual, predicted, coverage=0.9)
    interval = interval_metrics(actual, predicted, radius)
    assert 0 <= interval["empirical_coverage"] <= 1
    print("Energy forecasting self-test passed.")


def check_evidence() -> None:
    report = json.loads(EVIDENCE.read_text(encoding="utf-8"))
    assert report["verification_pass"] is True
    assert report["source_rows"] == 4383
    assert report["forecast_horizon_days"] == 14
    assert report["train_windows"] == 2995
    assert report["validation_windows"] == 644
    assert report["test_windows"] == 645
    model = report["test_metrics"]["tensorflow_lstm"]
    seasonal = report["test_metrics"]["weekly_seasonal"]
    assert model["mae_gwh"] < seasonal["mae_gwh"]
    assert report["mae_improvement_vs_best_baseline_pct"] > 18.0
    assert report["artifact"]["max_reload_prediction_delta"] == 0.0
    print("Retained energy forecast evidence passed.")


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--self-test", action="store_true")
    parser.add_argument("--check-evidence", action="store_true")
    args = parser.parse_args()
    if args.self_test:
        self_test()
    if args.check_evidence:
        check_evidence()
    if not args.self_test and not args.check_evidence:
        parser.error("choose --self-test or --check-evidence")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Canonical source: `src/__init__.py`


In [ ]:
"""Leakage-safe multi-horizon energy demand forecasting package."""


## Canonical source: `src/data.py`


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

DATA_URL = "https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv"
LOOKBACK = 60
HORIZON = 14


def load_daily_data(url: str = DATA_URL) -> pd.DataFrame:
    frame = pd.read_csv(url, parse_dates=["Date"]).sort_values("Date").set_index("Date")
    frame = frame.rename(
        columns={
            "Consumption": "consumption",
            "Wind": "wind",
            "Solar": "solar",
            "Wind+Solar": "wind_solar",
        }
    )
    if len(frame) != 4383:
        raise ValueError(f"expected 4,383 daily records, found {len(frame):,}")
    if not frame.index.is_monotonic_increasing or not frame.index.is_unique:
        raise ValueError("daily index must be sorted and unique")
    expected = pd.date_range(frame.index.min(), frame.index.max(), freq="D")
    if len(expected.difference(frame.index)):
        raise ValueError("source contains missing calendar dates")
    if frame["consumption"].isna().any() or (frame["consumption"] <= 0).any():
        raise ValueError("consumption target must be complete and positive")
    return frame


def chronological_windows(frame: pd.DataFrame):
    """Fit scaling on train only, then create 60-day -> 14-day windows."""
    n = len(frame)
    train_end = int(0.70 * n)
    validation_end = int(0.85 * n)
    values = frame[["consumption"]].to_numpy(dtype="float32")
    scaler = StandardScaler().fit(values[:train_end])
    scaled = scaler.transform(values).astype("float32")

    def make_windows(first_target: int, last_target: int):
        X, y, dates = [], [], []
        for index in range(max(first_target, LOOKBACK), last_target - HORIZON + 1):
            X.append(scaled[index - LOOKBACK:index])
            y.append(scaled[index:index + HORIZON, 0])
            dates.append(frame.index[index])
        return np.asarray(X), np.asarray(y), np.asarray(dates)

    train = make_windows(0, train_end)
    validation = make_windows(train_end, validation_end)
    test = make_windows(validation_end, n)
    if not (train[2].max() < validation[2].min() < test[2].min()):
        raise AssertionError("window chronology is invalid")
    return scaler, train, validation, test


def invert_scale(scaler: StandardScaler, values) -> np.ndarray:
    array = np.asarray(values)
    return scaler.inverse_transform(array.reshape(-1, 1)).reshape(array.shape)


## Canonical source: `src/evaluation.py`


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error


def forecast_metrics(actual, predicted) -> dict[str, float]:
    y = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    return {
        "mae": float(mean_absolute_error(y.ravel(), p.ravel())),
        "rmse": float(mean_squared_error(y.ravel(), p.ravel()) ** 0.5),
        "mape_pct": float(np.mean(np.abs((y - p) / y)) * 100),
    }


def last_value_baseline(X, horizon: int = 14) -> np.ndarray:
    history = np.asarray(X)
    return np.repeat(history[:, -1, 0, None], horizon, axis=1)


def seasonal_weekly_baseline(X, lookback: int = 60, horizon: int = 14) -> np.ndarray:
    history = np.asarray(X)
    return np.stack([history[:, lookback - 7 + (step % 7), 0] for step in range(horizon)], axis=1)


def calibrate_residual_intervals(actual_validation, predicted_validation, coverage: float = 0.90) -> np.ndarray:
    """Per-horizon absolute-residual quantiles fitted on validation only."""
    if not 0 < coverage < 1:
        raise ValueError("coverage must be between zero and one")
    residual = np.abs(np.asarray(actual_validation) - np.asarray(predicted_validation))
    return np.quantile(residual, coverage, axis=0, method="higher")


def interval_metrics(actual, predicted, radius) -> dict[str, float]:
    y = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    q = np.asarray(radius, dtype=float)
    lower = p - q
    upper = p + q
    return {
        "empirical_coverage": float(np.mean((y >= lower) & (y <= upper))),
        "average_width": float(np.mean(upper - lower)),
    }


## Canonical source: `src/model.py`


In [ ]:
from __future__ import annotations

LOOKBACK = 60
HORIZON = 14


def build_model():
    import tensorflow as tf

    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input((LOOKBACK, 1)),
            tf.keras.layers.Conv1D(32, 5, padding="causal", activation="relu"),
            tf.keras.layers.LSTM(48, dropout=0.15),
            tf.keras.layers.Dense(48, activation="relu"),
            tf.keras.layers.Dropout(0.15),
            tf.keras.layers.Dense(HORIZON),
        ],
        name="energy_forecaster",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mae",
        metrics=["mae"],
    )
    return model


def callbacks():
    import tensorflow as tf

    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            min_delta=1e-4,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            patience=3,
            factor=0.5,
            min_lr=1e-5,
        ),
    ]


## Canonical source: `tests/test_evaluation.py`


In [ ]:
import numpy as np

from src.evaluation import (
    calibrate_residual_intervals,
    forecast_metrics,
    interval_metrics,
    last_value_baseline,
    seasonal_weekly_baseline,
)


def test_baseline_shapes_and_alignment():
    X = np.arange(3 * 60, dtype=float).reshape(3, 60, 1) + 1.0
    last = last_value_baseline(X)
    weekly = seasonal_weekly_baseline(X)
    assert last.shape == (3, 14)
    assert weekly.shape == (3, 14)
    assert np.array_equal(weekly[:, 0], X[:, 53, 0])
    assert np.array_equal(weekly[:, 7], X[:, 53, 0])


def test_perfect_forecast_metrics():
    y = np.array([[100.0, 110.0], [120.0, 130.0]])
    metrics = forecast_metrics(y, y)
    assert metrics["mae"] == 0.0
    assert metrics["rmse"] == 0.0
    assert metrics["mape_pct"] == 0.0


def test_validation_calibrated_interval_has_valid_coverage():
    actual_val = np.array([[10.0, 20.0], [11.0, 19.0], [12.0, 18.0]])
    pred_val = np.array([[9.0, 18.0], [10.0, 20.0], [13.0, 17.0]])
    radius = calibrate_residual_intervals(actual_val, pred_val, coverage=0.9)
    test = interval_metrics(actual_val, pred_val, radius)
    assert radius.shape == (2,)
    assert 0.0 <= test["empirical_coverage"] <= 1.0
    assert test["average_width"] >= 0.0


# Portfolio depth check

**Meaningful visible code lines after all notebook passes:** 898. The working target for a major application is roughly 1,000 meaningful lines when justified by the problem. This notebook is in/above the working depth range. Line count is never permission to add filler; depth must come from data, analysis, visualisation, modelling/engineering, evaluation, robustness and decision logic.
